### Generate & scraping json data

Automating the process of scraping data from Wikipedia and generating data in JSON files because I'm lazy.


#### Imports


In [ ]:
import json
import requests
import pandas as pd

from io import StringIO
from bs4 import BeautifulSoup
from dotenv import dotenv_values

#### Look-ups


In [2]:
with open("../data/groups.json", "r") as f:
    groups_data = json.load(f)
    f.close()

with open("../data/teams.json", "r") as f:
    teams_data = json.load(f)
    f.close()

In [3]:
groups_lookup = pd.DataFrame(groups_data["groups"])
teams_lookup = pd.DataFrame(teams_data["teams"])

#### Create json data


##### Standings


In [ ]:
standings = []

for group in groups_lookup["name"]:
    request = requests.get(
        f"https://en.wikipedia.org/api/rest_v1/page/html/2026_FIFA_World_Cup_{group.replace(' ', '_')}",
        headers={
            "User-Agent": f"World-Cup-2026-App/1.0 ({dotenv_values('../.env')['EMAIL']})"
        },
    )

    soup = BeautifulSoup(request.text, "html.parser")
    table = pd.read_html(StringIO(str(soup.find_all("table", {"class": "wikitable"}))))[
        1
    ]

    # Remove host tag from team name
    table["Teamvte"] = table["Teamvte"].str.replace(r"\s*\(H\)", "", regex=True)

    # Add teamId and short team name
    team_ids = []
    short_names = []
    flags = []
    for team in table["Teamvte"]:
        team_info = teams_lookup[
            (teams_lookup["fullName"] == team) | (teams_lookup["shortName"] == team)
        ].reset_index(drop=True)
        # General cases
        if not team_info.empty:
            team_ids.append(team_info.loc[0, "id"] if len(team_info) > 0 else None)
            short_names.append(
                team_info.loc[0, "shortName"].values if len(team_info) > 0 else None
            )
            flags.append(
                team_info.loc[0, "flag"].values if len(team_info) > 0 else None
            )
        # Special cases
        else:
            # Wikipedia name - FIFA-recognised name
            special_cases = {
                "Cape Verde": "Cabo Verde",
                "DR Congo": "Congo DR",
                "Turkey": "Türkiye",
            }
            team_info = teams_lookup[
                teams_lookup["fullName"] == special_cases.get(team, team)
            ].reset_index(drop=True)
            team_ids.append(team_info.loc[0, "id"] if len(team_info) > 0 else None)
            short_names.append(
                team_info.loc[0, "shortName"] if len(team_info) > 0 else None
            )
            flags.append(team_info.loc[0, "flag"] if len(team_info) > 0 else None)

    # Add teamId column to table
    table["id"] = pd.Series(team_ids)
    table["shortName"] = pd.Series(short_names)
    table["flag"] = pd.Series(flags)

    # Some pre-processing steps
    ## Rename columns
    table.rename(
        columns={
            "Teamvte": "teamName",
            "Pos": "position",
            "Pld": "played",
            "W": "wins",
            "D": "draws",
            "L": "losses",
            "GF": "goalsScored",
            "GA": "goalsConceded",
            "GD": "goalDiff",
            "Pts": "points",
        },
        inplace=True,
    )

    ## Drop unnecessary columns
    table.drop(columns=["Qualification"], inplace=True, errors="ignore")

    ## Move id column in front of teamName
    cols = table.columns.tolist()
    cols.insert(0, cols.pop(cols.index("id")))
    table = table[cols]

    # Convert to dict and add to standings
    standings.append(
        {
            "id": groups_lookup[groups_lookup["name"] == group]["id"].iloc[0],
            "name": group,
            "teams": table.to_dict(orient="records"),
        }
    )

                 id fullName shortName code primaryColor textColor  \
1  c5f0e4g73d6b7h27   Mexico    Mexico  MEX      #27A550   #000000   

                                                flag  \
1  https://cdn.countryflags.com/thumbs/mexico/fla...   

                                               shirt  
1  https://play.fifa.com/media/image/fantasy/squa...  
                 id fullName shortName code primaryColor textColor  \
0  a3f8c2d91b4e7f05   Canada    Canada  CAN      #D52B1E   #FFFFFF   

                                                flag  \
0  https://cdn.countryflags.com/thumbs/canada/fla...   

                                               shirt  
0  https://play.fifa.com/media/image/fantasy/squa...  
                 id fullName shortName code primaryColor textColor  \
9  j2g7m1h69i0e4l18   Brazil    Brazil  BRA      #FFCF25   #000000   

                                                flag  \
9  https://cdn.countryflags.com/thumbs/brazil/fla...   

                  

In [21]:
# Write standings to JSON file
with open("../data/standings.json", "w", encoding="utf-8") as f:
    json.dump({"standings": standings}, f, indent=3, ensure_ascii=False)
    f.close()

TypeError: Object of type StringArray is not JSON serializable

##### Matches/Results


##### Squads
